<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/languages/python/mini_projects/experiment_terminal_dungeon_adventure_game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import random
import time
from dataclasses import dataclass
from enum import Enum
from IPython.display import display, HTML, clear_output
from google.colab import output # New import for Colab interactivity

# --- Global game instance (for callback context) ---
game_instance = None

# --- Enums for Game State and Map Tiles ---
class GameState(Enum):
    PLAYING = 1
    GAME_OVER_LOSE = 2
    GAME_OVER_WIN = 3

class TileType(Enum):
    WALL = '#'
    FLOOR = '.'
    PLAYER = '@'
    ENEMY = 'E'
    TREASURE = 'T'
    EXIT = 'X'

# --- Dataclasses for Game Entities ---
@dataclass
class Position:
    x: int
    y: int

@dataclass
class Player:
    position: Position
    health: int = 100
    gold: int = 0
    attack: int = 15

@dataclass
class Enemy:
    id: int
    position: Position
    health: int = 30
    attack: int = 10

@dataclass
class Treasure:
    id: int
    position: Position
    value: int = 20

# --- Game Class ---
class DungeonCrawlerGame:
    def __init__(self, width=15, height=10, num_enemies=3, num_treasures=3):
        self.width = width
        self.height = height
        self.player = Player(position=Position(0, 0))
        self.enemies = []
        self.treasures = []
        self.exit_pos = Position(0, 0)
        self.game_state = GameState.PLAYING
        self.message = "Welcome to the Dungeon!"
        self._generate_map(num_enemies, num_treasures)

    def _generate_map(self, num_enemies, num_treasures):
        # Initialize map with walls, then create a path for floor
        self.game_map = [[TileType.WALL for _ in range(self.width)] for _ in range(self.height)]

        # Simple room generation (for now, just make most of it floor)
        for y in range(1, self.height - 1):
            for x in range(1, self.width - 1):
                self.game_map[y][x] = TileType.FLOOR

        # Place entities
        self._place_entities(num_enemies, num_treasures)

    def _get_random_floor_position(self):
        while True:
            x = random.randint(1, self.width - 2)
            y = random.randint(1, self.height - 2)
            if self.game_map[y][x] == TileType.FLOOR:
                return Position(x, y)

    def _place_entities(self, num_enemies, num_treasures):
        # Place player
        self.player.position = self._get_random_floor_position()

        # Place exit
        self.exit_pos = self._get_random_floor_position()
        # Ensure player and exit don't start on same spot
        while self.exit_pos == self.player.position:
            self.exit_pos = self._get_random_floor_position()

        # Place enemies
        for i in range(num_enemies):
            pos = self._get_random_floor_position()
            while pos == self.player.position or pos == self.exit_pos or any(e.position == pos for e in self.enemies):
                pos = self._get_random_floor_position()
            self.enemies.append(Enemy(id=i, position=pos))

        # Place treasures
        for i in range(num_treasures):
            pos = self._get_random_floor_position()
            while pos == self.player.position or pos == self.exit_pos or any(t.position == pos for t in self.treasures) or any(e.position == pos for e in self.enemies):
                pos = self._get_random_floor_position()
            self.treasures.append(Treasure(id=i, position=pos))

    def _get_tile_at(self, pos: Position):
        if 0 <= pos.y < self.height and 0 <= pos.x < self.width:
            return self.game_map[pos.y][pos.x]
        return TileType.WALL # Out of bounds is a wall

    def _move_player(self, dx, dy):
        if self.game_state != GameState.PLAYING:
            return

        new_x = self.player.position.x + dx
        new_y = self.player.position.y + dy
        new_pos = Position(new_x, new_y)

        if self._get_tile_at(new_pos) == TileType.WALL:
            self.message = "You hit a wall!"
            return

        self.player.position = new_pos
        self.message = "You move."

        # Check for interactions
        self._check_interactions(new_pos)

    def _check_interactions(self, pos: Position):
        # Check for exit
        if pos == self.exit_pos:
            self.game_state = GameState.GAME_OVER_WIN
            self.message = "You found the exit! You win!"
            return

        # Check for treasures
        for treasure in self.treasures:
            if treasure.position == pos:
                self.player.gold += treasure.value
                self.treasures.remove(treasure)
                self.message = f"You found {treasure.value} gold!"
                break

        # Check for enemies (only if not already won/lost)
        if self.game_state == GameState.PLAYING:
            for enemy in self.enemies:
                if enemy.position == pos:
                    self._handle_enemy_encounter(enemy)
                    break

    def _handle_enemy_encounter(self, enemy: Enemy):
        self.message = f"You encountered an enemy! Health: {enemy.health}"
        self._display_game_state() # Update display for encounter message
        time.sleep(1) # Pause before combat starts

        while enemy.health > 0 and self.player.health > 0:
            # Player attacks
            enemy.health -= self.player.attack
            self.message = f"You hit the enemy for {self.player.attack}! Enemy health: {enemy.health}"
            if enemy.health <= 0:
                self.message += " You defeated the enemy!"
                self.enemies.remove(enemy)
                break
            # Enemy attacks
            self.player.health -= enemy.attack
            self.message += f" The enemy hits you for {enemy.attack}! Your health: {self.player.health}"
            if self.player.health <= 0:
                self.game_state = GameState.GAME_OVER_LOSE
                self.message += " You were defeated! Game Over!"
                break
            self._display_game_state()
            time.sleep(1) # Pause briefly during combat

    def _render_map(self):
        display_map = [row[:] for row in self.game_map] # Copy map

        if self.game_state == GameState.PLAYING or self.game_state == GameState.GAME_OVER_WIN:
             # Always show the exit if game is ongoing or won
             display_map[self.exit_pos.y][self.exit_pos.x] = TileType.EXIT

        if self.game_state == GameState.PLAYING:
            # Only show entities if the game is still playing
            for treasure in self.treasures:
                display_map[treasure.position.y][treasure.position.x] = TileType.TREASURE
            for enemy in self.enemies:
                display_map[enemy.position.y][enemy.position.x] = TileType.ENEMY

        # Player is always drawn last to ensure visibility if on same tile as other entity
        display_map[self.player.position.y][self.player.position.x] = TileType.PLAYER

        map_str = "" # Use an empty string for efficient concatenation
        for row in display_map:
            map_str += " ".join([tile.value for tile in row]) + "\n"
        return map_str

    def _render_controls(self):
        # Using a single callback 'game_action_callback' to receive button clicks
        html_buttons = f"""
        <div style="margin-top: 10px; text-align: center;">
            <button onclick="game_input('w')" style="width: 50px; height: 30px; margin: 5px;">W</button><br>
            <button onclick="game_input('a')" style="width: 50px; height: 30px; margin: 5px;">A</button>
            <button onclick="game_input('s')" style="width: 50px; height: 30px; margin: 5px;">S</button>
            <button onclick="game_input('d')" style="width: 50px; height: 30px; margin: 5px;">D</button><br>
            <button onclick="game_input('q')" style="margin-top: 10px; background-color: #ffcccc; width: 150px; height: 30px;">Quit</button>
        </div>
        <script>
            // Define a function that the buttons will call
            function game_input(action) {{
                // Invoke the Python callback registered with 'game_action_callback'
                google.colab.kernel.invokeFunction('game_action_callback', [action]);
            }}
        </script>
        """
        display(HTML(html_buttons))

    def _display_game_state(self):
        clear_output(wait=True)
        print(self._render_map())
        print(f"--- Player Stats ---")
        print(f"Health: {self.player.health} | Gold: {self.player.gold}")
        print(f"Message: {self.message}")

        if self.game_state == GameState.PLAYING:
            self._render_controls() # Display buttons only when playing
        else:
            print(f"GAME OVER! {self.game_state.name.replace('_', ' ')}!")
            print("Thank you for playing!")

    def play(self):
        global game_instance
        game_instance = self # Set the global instance for callbacks

        # Register the Python function that JavaScript buttons will call
        output.register_callback('game_action_callback', self._process_action)

        self._display_game_state() # Initial display of the game and controls


    def _process_action(self, action):
        # This function is called by the JavaScript buttons via the registered callback
        if self.game_state != GameState.PLAYING:
            # If the game is already over, ignore further input
            self.message = "Game is over!"
            self._display_game_state()
            return

        if action == 'w':
            self._move_player(0, -1)
        elif action == 's':
            self._move_player(0, 1)
        elif action == 'a':
            self._move_player(-1, 0)
        elif action == 'd':
            self._move_player(1, 0)
        elif action == 'q':
            self.game_state = GameState.GAME_OVER_LOSE # Player quits
            self.message = "You quit the game. Game Over!"
        else:
            self.message = "Invalid action. Please use the provided buttons."

        self._display_game_state() # Re-render the game state after each action


# --- Game Execution ---
print("Initializing Dungeon Crawler Game...")
# Ensure the game is re-initialized if the cell is run again
if 'game_instance' in globals() and game_instance:
    del game_instance
game = DungeonCrawlerGame()
game.play()


# # # # # # # # # # # # # # #
# . . . . . . . . . . . . . #
# . . . . . . . . . . . . . #
# . . . . . . . . . . . @ . #
# . . . . . . . . . . . . . #
# . . . . . . . . . . . . . #
# . . . . . . . . . . . . . #
# . . . . . . . . . . . . . #
# . . . . . . . . . . . . . #
# # # # # # # # # # # # # # #

--- Player Stats ---
Health: 100 | Gold: 20
Message: You found the exit! You win!
GAME OVER! GAME OVER WIN!
Thank you for playing!
